# 第10章：集成学习与模型融合

## 本章学习目标

- 理解集成学习方法
- 掌握模型融合策略
- 学会特征重要性分析
- 能够实现多模型集成系统

---

## 10.1 集成学习概述

集成学习通过组合多个模型来提升预测性能。

### 集成方法分类

```
集成学习方法
├── Bagging (并行)
│   └── 随机森林、LightGBM 内部集成
├── Boosting (串行)
│   └── GBDT、XGBoost、AdaBoost
└── Stacking (分层)
    └── 元模型组合多个基模型
```

### 融合策略

| 策略 | 方法 | 优点 |
|------|------|------|
| 简单平均 | 预测值直接平均 | 简单稳定 |
| 加权平均 | 根据性能加权 | 灵活调整 |
| Rank 融合 | 排名值平均 | 减少异常值影响 |
| Stacking | 元模型学习 | 自适应组合 |

In [ ]:
import qlib
from qlib.data.dataset import DatasetH, TSDatasetH
from qlib.contrib.data.handler import Alpha158, Alpha360
from qlib.contrib.model.gbdt import LGBModel
from qlib.workflow import R
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 10.2 训练多个基础模型

In [ ]:
# 创建数据集
# 注意：cn_data 数据最晚到 2020-09-25
dataset = DatasetH(
    handler={
        "class": "Alpha158",
        "module_path": "qlib.contrib.data.handler",
        "kwargs": {
            "start_time": "2015-01-01",
            "end_time": "2020-09-25",  # 数据最晚日期
            "fit_start_time": "2015-01-01",
            "fit_end_time": "2018-12-31",
            "instruments": "csi300",
        },
    },
    segments={
        "train": ("2015-01-01", "2018-12-31"),
        "valid": ("2019-01-01", "2020-06-30"),
        "test": ("2020-07-01", "2020-09-25"),  # 数据最晚日期
    },
)

print("数据集创建完成")

In [ ]:
# 训练多个 LightGBM 模型（不同配置）
model_configs = {
    "lgb_shallow": {
        "num_leaves": 32,
        "max_depth": 4,
        "learning_rate": 0.05,
        "n_estimators": 300,
    },
    "lgb_medium": {
        "num_leaves": 64,
        "max_depth": 6,
        "learning_rate": 0.05,
        "n_estimators": 500,
    },
    "lgb_deep": {
        "num_leaves": 128,
        "max_depth": 8,
        "learning_rate": 0.03,
        "n_estimators": 700,
    },
}

# 存储模型和预测结果
models = {}
predictions = {}

for name, config in model_configs.items():
    print(f"\n训练模型: {name}")
    
    model = LGBModel(
        loss="mse",
        colsample_bytree=0.8,
        subsample=0.8,
        random_state=42,
        n_jobs=4,
        **config,
    )
    
    model.fit(dataset)
    pred = model.predict(dataset)
    
    models[name] = model
    predictions[name] = pred
    
    print(f"  训练完成")

print("\n所有模型训练完成")

In [ ]:
# 评估各模型
def evaluate_predictions(predictions, labels):
    """评估预测结果"""
    pred = np.array(predictions).ravel()
    label = np.array(labels).ravel()
    
    mask = ~(np.isnan(pred) | np.isnan(label))
    pred = pred[mask]
    label = label[mask]
    
    ic = np.corrcoef(pred, label)[0, 1]
    rank_ic = np.corrcoef(np.argsort(np.argsort(pred)), np.argsort(np.argsort(label)))[0, 1]
    
    return {"IC": ic, "Rank IC": rank_ic}

test_data = dataset.prepare("test")
labels = test_data['label']

# 评估每个模型
results = {}
for name, pred in predictions.items():
    results[name] = evaluate_predictions(pred, labels)

results_df = pd.DataFrame(results).T
print("各模型评估结果:")
results_df

## 10.3 模型融合策略

### 10.3.1 简单平均融合

In [ ]:
# 简单平均融合
def simple_average(predictions_dict):
    """简单平均融合"""
    pred_list = list(predictions_dict.values())
    avg_pred = sum(pred_list) / len(pred_list)
    return avg_pred

# 执行融合
avg_predictions = simple_average(predictions)
avg_metrics = evaluate_predictions(avg_predictions, labels)

print("简单平均融合结果:")
for k, v in avg_metrics.items():
    print(f"  {k}: {v:.4f}")

### 10.3.2 加权平均融合

In [ ]:
# 加权平均融合（根据验证集 IC 确定权重）
def weighted_average(predictions_dict, weights):
    """加权平均融合"""
    total_weight = sum(weights.values())
    weighted_pred = sum(pred * weights[name] / total_weight 
                        for name, pred in predictions_dict.items())
    return weighted_pred

# 使用验证集 IC 作为权重
valid_data = dataset.prepare("valid")
valid_labels = valid_data['label']

weights = {}
for name, pred in predictions.items():
    # 在验证集上评估
    valid_pred = pred.loc[pred.index.isin(valid_data.index)]
    valid_labels_subset = valid_labels.loc[valid_labels.index.isin(valid_pred.index)]
    metrics = evaluate_predictions(valid_pred, valid_labels_subset)
    weights[name] = max(metrics['IC'], 0)  # 只使用正权重

print("模型权重:")
for name, w in weights.items():
    print(f"  {name}: {w:.4f}")

# 执行加权融合
weighted_pred = weighted_average(predictions, weights)
weighted_metrics = evaluate_predictions(weighted_pred, labels)

print("\n加权平均融合结果:")
for k, v in weighted_metrics.items():
    print(f"  {k}: {v:.4f}")

### 10.3.3 Rank 融合

In [ ]:
# Rank 融合
def rank_fusion(predictions_dict):
    """Rank 融合：对每个模型的预测排名后取平均"""
    rank_sum = None
    
    for name, pred in predictions_dict.items():
        # 计算排名
        rank = pred.groupby(level='datetime').rank(pct=True)
        
        if rank_sum is None:
            rank_sum = rank
        else:
            rank_sum = rank_sum + rank
    
    # 平均排名
    avg_rank = rank_sum / len(predictions_dict)
    return avg_rank

# 执行 Rank 融合
rank_predictions = rank_fusion(predictions)
rank_metrics = evaluate_predictions(rank_predictions, labels)

print("Rank 融合结果:")
for k, v in rank_metrics.items():
    print(f"  {k}: {v:.4f}")

## 10.4 融合效果对比

In [ ]:
# 汇总所有结果
all_results = {
    **results,
    "简单平均": avg_metrics,
    "加权平均": weighted_metrics,
    "Rank融合": rank_metrics,
}

all_results_df = pd.DataFrame(all_results).T

print("所有方法评估结果对比:")
all_results_df.sort_values('IC', ascending=False)

In [ ]:
# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IC 对比
all_results_df['IC'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('IC 对比')
axes[0].set_xlabel('IC')
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# Rank IC 对比
all_results_df['Rank IC'].sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Rank IC 对比')
axes[1].set_xlabel('Rank IC')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 10.5 特征重要性分析 (SHAP)

In [ ]:
# 使用 SHAP 分析特征重要性
# 注意：需要安装 shap 库

try:
    import shap
    
    # 获取模型和特征数据
    best_model = models['lgb_medium']
    test_data = dataset.prepare("test")
    X_test = test_data['feature']
    
    # 使用 TreeExplainer
    explainer = shap.TreeExplainer(best_model.model)
    
    # 计算 SHAP 值（可能较慢，采样部分数据）
    sample_size = min(1000, len(X_test))
    X_sample = X_test.sample(n=sample_size, random_state=42)
    
    shap_values = explainer.shap_values(X_sample)
    
    # 可视化
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample, show=False, max_display=20)
    plt.title('SHAP 特征重要性')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("未安装 shap 库，跳过 SHAP 分析")
    print("安装方法: pip install shap")

## 10.6 使用 qlib 内置集成方法

In [ ]:
# 查看 qlib 内置的集成方法
from qlib.model.ens import ensemble

print("Qlib 内置集成方法:")
print("=" * 50)

# 查看可用的集成类
import inspect
for name, obj in inspect.getmembers(ensemble):
    if inspect.isclass(obj):
        print(f"  - {name}")

In [ ]:
# 使用 SingleKeyEnsemble
from qlib.model.ens.ensemble import SingleKeyEnsemble

# SingleKeyEnsemble 用于将多个 recorder 的预测结果融合
print("SingleKeyEnsemble 使用说明:")
print("=" * 50)
print("\n1. 保存每个模型的预测结果到 Recorder")
print("2. 使用 SingleKeyEnsemble 聚合多个 Recorder 的结果")
print("3. 支持的聚合方法: mean, median, max, min")

## 10.7 完整集成流程示例

In [ ]:
# 完整的集成学习流程
with R.start(experiment_name="ensemble_experiment") as recorder:
    
    # 记录各模型参数
    recorder.log_params({
        "models": list(model_configs.keys()),
        "fusion_method": "weighted_average",
    })
    
    # 记录各模型性能
    for name, metrics in results.items():
        recorder.log_metrics({f"{name}_IC": metrics['IC']})
    
    # 记录融合性能
    recorder.log_metrics({
        "simple_avg_IC": avg_metrics['IC'],
        "weighted_avg_IC": weighted_metrics['IC'],
        "rank_fusion_IC": rank_metrics['IC'],
    })
    
    # 保存最佳融合预测
    best_method = max(['simple_avg', 'weighted_avg', 'rank_fusion'],
                      key=lambda x: {'simple_avg': avg_metrics['IC'],
                                     'weighted_avg': weighted_metrics['IC'],
                                     'rank_fusion': rank_metrics['IC']}[x])
    
    print(f"最佳融合方法: {best_method}")
    print(f"\nRecorder ID: {recorder.id}")

## 10.8 实践练习

In [ ]:
# 练习1: 实现一个 Stacking 融合方法
# 使用线性回归作为元模型

# 你的代码



# 参考答案
# from sklearn.linear_model import Ridge
# 
# # 准备训练数据
# train_data = dataset.prepare("train")
# train_labels = train_data['label']
# 
# # 获取各模型在训练集的预测
# train_preds = []
# for name, model in models.items():
#     train_pred = model.predict(dataset)
#     train_preds.append(train_pred.loc[train_data.index])
# 
# # 构建元特征
# X_meta = pd.concat(train_preds, axis=1)
# X_meta.columns = list(models.keys())
# 
# # 训练元模型
# meta_model = Ridge()
# meta_model.fit(X_meta, train_labels)
# 
# # 预测
# test_preds = []
# for name, pred in predictions.items():
#     test_preds.append(pred)
# X_test_meta = pd.concat(test_preds, axis=1)
# X_test_meta.columns = list(models.keys())
# stacking_pred = meta_model.predict(X_test_meta)

In [ ]:
# 练习2: 尝试更多模型组合
# 添加 LSTM 模型到集成中

# 你的代码



# 提示：参考第9章的 LSTM 训练代码

In [ ]:
# 练习3: 分析不同融合方法的稳定性
# 使用滑动窗口验证融合效果的一致性

# 你的代码



# 提示：将测试集分成多个时间段，分别计算各方法的 IC

## 10.9 本章小结

本章我们学习了：

1. **集成学习方法**：
   - Bagging、Boosting、Stacking
   - 多模型训练

2. **融合策略**：
   - 简单平均
   - 加权平均
   - Rank 融合
   - Stacking

3. **特征重要性**：
   - SHAP 分析
   - 特征贡献解读

4. **最佳实践**：
   - 模型多样性
   - 验证集权重选择
   - 稳定性分析

### 融合方法对比

| 方法 | 复杂度 | 效果 | 适用场景 |
|------|--------|------|----------|
| 简单平均 | 低 | 中 | 基线方法 |
| 加权平均 | 低 | 中高 | 模型性能差异大 |
| Rank 融合 | 低 | 中 | 异常值敏感 |
| Stacking | 高 | 高 | 追求最优性能 |

### 下一部分预告

下一部分我们将学习策略与回测，包括：
- 策略框架
- 回测系统
- 回测报告与分析